In [1]:
!pip install transformers[torch] datasets sentencepiece accelerate -U

import torch
from datasets import load_dataset
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.5/527.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 88.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [3]:
# 1. Install necessary libraries
!pip install transformers[torch] datasets sentencepiece accelerate -U

# 2. Log in to Hugging Face
from huggingface_hub import login
# Replace 'your_token_here' with the actual token from your screenshot
login("quiz_Token")

In [4]:
from datasets import load_dataset
from transformers import T5Tokenizer

# Load 5000 samples of the SQuAD dataset for training
dataset = load_dataset("squad", split='train[:5000]')
tokenizer = T5Tokenizer.from_pretrained("t5-small")

def preprocess_function(examples):
    # Prepare the input: Context + Answer
    inputs = ["context: " + c + " answer: " + a['text'][0] for c, a in zip(examples['context'], examples['answers'])]

    # Tokenize inputs
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding="max_length")

    # Tokenize targets (The Questions) using the new 'text_target' parameter
    labels = tokenizer(text_target=examples['question'], max_length=64, truncation=True, padding="max_length")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Apply the preprocessing
tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [6]:
from transformers import T5ForConditionalGeneration, Trainer, TrainingArguments

# Load the pre-trained T5-small model
model = T5ForConditionalGeneration.from_pretrained("t5-small")

# Define training parameters
training_args = TrainingArguments(
    output_dir="./quiz_model_results",
    eval_strategy="no",        # FIXED: Changed from evaluation_strategy to eval_strategy
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    save_total_limit=2,
    fp16=True,
    report_to="none"
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

In [7]:
# Start the fine-tuning process
trainer.train()

# Save the model and tokenizer
model.save_pretrained("./final_quiz_model")
tokenizer.save_pretrained("./final_quiz_model")

print("Training complete!")

Step,Training Loss
500,0.573981


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [8]:
import shutil
from google.colab import files

# Zip the folder
shutil.make_archive("quiz_model_export", 'zip', "./final_quiz_model")

# Download the zip file
files.download("quiz_model_export.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [9]:
def generate_quiz_question(context, answer):
    # Format the input just like we did in training
    input_text = f"context: {context} answer: {answer}"

    # Encode and move to GPU
    inputs = tokenizer(input_text, return_tensors="pt").to("cuda")

    # Generate the question
    outputs = model.generate(inputs["input_ids"], max_length=64)

    # Decode and return
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- TEST IT HERE ---
my_context = "FastAPI is a modern, fast (high-performance), web framework for building APIs with Python 3.8+."
my_answer = "Python 3.8+"

question = generate_quiz_question(my_context, my_answer)
print(f"Topic: FastAPI\nAnswer Provided: {my_answer}\nGenerated Question: {question}")

Topic: FastAPI
Answer Provided: Python 3.8+
Generated Question: What is the name of the APIs that are built with?


In [ ]:
from google.colab import drive
drive.mount('/content/drive')